# Análisis NLP de Discursos Presidenciales Chilenos

---

## Introducción

Este cuaderno presenta un **análisis computacional del discurso político chileno** basado en las Cuentas Públicas presidenciales abarcando un período histórico de casi dos siglos.

### Objetivo

Extraer patrones temáticos, entidades nombradas y tendencias discursivas utilizando técnicas de **Procesamiento de Lenguaje Natural (NLP)**:

| Técnica | Descripción |
|---------|-------------|
| **Tokenización y limpieza** | Normalización del texto, eliminación de stopwords en español |
| **Análisis de bigramas** | Identificación de pares de palabras más frecuentes |
| **Modelado de tópicos** | Latent Dirichlet Allocation para descubrir temas latentes |
| **Nube de palabras** | Visualización intuitiva de frecuencias léxicas |
| **Distribución temática** | Evolución de temas por presidente y período |

### Datos

El corpus incluye discursos de **14 presidentes** entre **1842 y 2000**, representando momentos clave de la historia chilena:
- Dictadura, república, guerra, terremotos
- Transiciones democráticas y reformas constitucionales
- Desarrollo económico y social del siglo XX

---

## 2. Carga de Datos

Cargamos el corpus de discursos presidenciales en formato CSV, donde cada registro contiene el **año**, el **presidente** y el **texto completo** del discurso.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import re
import json
from pathlib import Path
from collections import Counter

BASE = Path('..')
csv_path = BASE / 'data' / 'sample_speeches.csv'
if not csv_path.exists():
    csv_path = BASE / 'data' / 'processed' / 'speeches.csv'

df = pd.read_csv(csv_path)

print(f'Discursos cargados: {len(df)}')
print(f'Presidentes: {df["speaker"].nunique()}')
print(f'Período: {int(df["year"].min())} - {int(df["year"].max())}')
df.head()

In [ ]:
df['word_count'] = df['text'].str.split().str.len()
df['char_count'] = df['text'].str.len()
df['sentence_count'] = df['text'].str.count(r'[.!?]+')

print('=== Estadísticas del Corpus ===')
print(f'Total de palabras: {df["word_count"].sum():,.0f}')
print(f'Promedio por discurso: {df["word_count"].mean():,.0f}')
print(f'Discursos más extensos:')
print(df[['year', 'speaker', 'word_count']].sort_values('word_count', ascending=False).head())

## 3. Preprocesamiento del Texto

Implementamos el pipeline de limpieza siguiendo los patrones de `src/topic_analysis.py`:

1. **Normalización**: Conversión a minúsculas
2. **Limpieza**: Eliminación de caracteres especiales
3. **Filtrado**: Eliminación de stopwords y palabras cortas
4. **Tokenización**: Separación en tokens individuales

In [ ]:
STOPWORDS = [
    "de", "la", "el", "en", "y", "a", "que", "es", "se", "del", "los", "las",
    "un", "una", "por", "con", "no", "para", "al", "lo", "como", "su", "más",
    "este", "ha", "han", "han sido", "ha sido", "fue", "ser", "ha de", "sobre",
    "todo", "entre", "desde", "sin", "pero", "muy", "ya", "o", "e", "ni", "le",
    "les", "da", "dos", "cada", "uno", "otra", "otro", "otros", "otras",
]

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-záéíóúñü\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return ' '.join(words)

df['clean_text'] = df['text'].apply(clean_text)

print('=== Texto original vs limpio ===')
print(f'Original: {df["text"].iloc[0][:200]}...')
print(f'Limpio:   {df["clean_text"].iloc[0][:200]}...')
print(f'\nReducción promedio: {1 - df["clean_text"].str.split().str.len().mean() / df["word_count"].mean():.1%}')

In [ ]:
all_words = ' '.join(df['clean_text']).split()
word_freq = Counter(all_words)
top_words = word_freq.most_common(30)

words_df = pd.DataFrame(top_words, columns=['palabra', 'frecuencia'])

fig = px.bar(words_df, x='frecuencia', y='palabra', orientation='h',
             title='Top 30 Palabras Más Frecuentes en Discursos Presidenciales',
             labels={'frecuencia': 'Frecuencia', 'palabra': ''},
             color='frecuencia',
             color_continuous_scale=['#c5a55a', '#6b1d1d'])
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600,
                  paper_bgcolor='#faf0d7', plot_bgcolor='#f4e8c1',
                  font=dict(family='Georgia, serif'))
fig.show()

In [ ]:
try:
    from wordcloud import WordCloud
    import matplotlib.pyplot as plt

    wc = WordCloud(width=1000, height=500, background_color='#faf0d7',
                   colormap='copper', max_words=100,
                   contour_width=2, contour_color='#6b1d1d')
    wc.generate_from_frequencies(word_freq)

    fig_wc, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('Nube de Palabras — Discursos Presidenciales Chilenos',
                 fontsize=16, fontfamily='Georgia', color='#6b1d1d', pad=20)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('wordcloud no instalado. Ejecuta: pip install wordcloud matplotlib')
    print('Los análisis de frecuencia mostrados arriba son equivalentes.')

## 4. Análisis de Bigramas

Los bigramas (pares de palabras consecutivas) revelan **estructuras discursivas recurrentes** que las palabras aisladas no capturan. Por ejemplo, "libre mercado" o "derechos humanos" son bigramas que transmiten significados específicos.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import networkx as nx

vectorizer_bigrams = CountVectorizer(min_df=1, max_features=200, ngram_range=(2, 2))
bigram_matrix = vectorizer_bigrams.fit_transform(df['clean_text'])
bigram_names = vectorizer_bigrams.get_feature_names_out()
bigram_counts = bigram_matrix.sum(axis=0).A1

bigram_pairs = sorted(zip(bigram_names, bigram_counts), key=lambda x: -x[1])
bigram_df = pd.DataFrame(bigram_pairs, columns=['bigrama', 'frecuencia'])

print(f'Bigramas únicos encontrados: {len(bigram_pairs)}')
print(f'\nTop 15 bigramas:')
bigram_df.head(15)

In [ ]:
fig = px.bar(bigram_df.head(20), x='frecuencia', y='bigrama', orientation='h',
             title='Top 20 Bigramas en Discursos Presidenciales',
             labels={'frecuencia': 'Frecuencia', 'bigrama': ''},
             color='frecuencia',
             color_continuous_scale=['#c5a55a', '#6b1d1d'])
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=550,
                  paper_bgcolor='#faf0d7', plot_bgcolor='#f4e8c1',
                  font=dict(family='Georgia, serif'))
fig.show()

In [ ]:
top_bigrams = bigram_df.head(15)
G = nx.Graph()

for _, row in top_bigrams.iterrows():
    parts = row['bigrama'].split()
    if len(parts) == 2:
        G.add_edge(parts[0], parts[1], weight=row['frecuencia'])

pos = nx.spring_layout(G, k=2, seed=42)
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
max_weight = max(edge_weights) if edge_weights else 1

fig_network = go.Figure()

for u, v, w in G.edges(data=True):
    fig_network.add_trace(go.Scatter(
        x=[pos[u][0], pos[v][0]], y=[pos[u][1], pos[v][1]],
        mode='lines', line=dict(width=w['weight']/max_weight*6+1, color='#c5a55a'),
        showlegend=False, hoverinfo='text',
        text=f"{u} — {v}: {w['weight']}"
    ))

node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_degrees = [G.degree(n) for n in G.nodes()]
node_freq = [word_freq.get(n, 1) for n in G.nodes()]

fig_network.add_trace(go.Scatter(
    x=node_x, y=node_y, mode='markers+text',
    marker=dict(size=[f/3+10 for f in node_freq], color=node_degrees,
                colorscale='Burg', line=dict(width=1, color='#6b1d1d')),
    text=list(G.nodes()), textposition='top center',
    textfont=dict(family='Georgia, serif', size=10, color='#3a2a1a'),
    hovertext=[f"{n}<br>Frecuencia: {word_freq.get(n, 0)}<br>Conexiones: {G.degree(n)}" for n in G.nodes()],
    showlegend=False
))

fig_network.update_layout(
    title='Red de Co-ocurrencia de Bigramas',
    paper_bgcolor='#faf0d7', plot_bgcolor='#f4e8c1',
    font=dict(family='Georgia, serif'),
    showlegend=False, height=500,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
)
fig_network.show()

## 5. Modelado de Temas con LDA

Utilizamos **Latent Dirichlet Allocation (LDA)**, un modelo generativo que descubre topics latentes en colecciones de documentos. Cada topic es una distribución sobre palabras, y cada documento es una mezcla de topics.

- **n_components=5**: Número de temas a descubrir
- **max_iter=50**: Iteraciones de optimización
- **random_state=42**: Reproducibilidad

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

N_TOPICS = 5

vectorizer_words = CountVectorizer(min_df=1, max_features=500)
word_matrix = vectorizer_words.fit_transform(df['clean_text'])

lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=50)
doc_topics = lda.fit_transform(word_matrix)

feature_names = vectorizer_words.get_feature_names_out()

topics_data = []
print('=== Temas Descubiertos ===')
for idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]
    weight = round(float(topic.sum()), 2)
    topics_data.append({'topic_id': idx, 'words': top_words, 'weight': weight})
    print(f'\nTema {idx} (peso: {weight}):')
    print(f'  {" | ".join(top_words[:5])}')

In [ ]:
topic_labels = []
for t in topics_data:
    label = ', '.join(t['words'][:3])
    topic_labels.append(f"Tema {t['topic_id']}: {label}")

topic_words_data = []
for t in topics_data:
    for w in t['words'][:5]:
        topic_words_data.append({
            'tema': f"Tema {t['topic_id']}",
            'palabra': w
        })
topic_words_df = pd.DataFrame(topic_words_data)

fig = px.treemap(topic_words_df, path=['tema', 'palabra'],
                 title='Estructura de Temas: Palabras Clave por Tema LDA',
                 color='tema',
                 color_discrete_sequence=['#6b1d1d', '#c5a55a', '#2d5016', '#1a2744', '#8a2e2e'])
fig.update_layout(paper_bgcolor='#faf0d7', plot_bgcolor='#f4e8c1',
                  font=dict(family='Georgia, serif'), height=500)
fig.show()

In [ ]:
topic_weight_df = pd.DataFrame(topics_data)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Peso Total por Tema', 'Top Palabras por Tema'),
                    column_widths=[0.35, 0.65])

fig.add_trace(
    go.Bar(x=topic_weight_df['topic_id'], y=topic_weight_df['weight'],
           marker_color=['#c5a55a', '#6b1d1d', '#2d5016', '#1a2744', '#8a2e2e'],
           text=topic_weight_df['weight'], textposition='auto',
           name='Peso'),
    row=1, col=1
)

for i, t in enumerate(topics_data):
    words = t['words'][:5][::-1]
    weights = list(range(1, len(words)+1))
    fig.add_trace(
        go.Bar(x=weights, y=words, orientation='h',
               marker_color=['#c5a55a', '#b8944e', '#ab8342', '#9e7236', '#6b1d1d'][::-1],
               name=f'Tema {i}', showlegend=False),
        row=1, col=2
    )

fig.update_layout(title='Análisis de Temas LDA',
                  paper_bgcolor='#faf0d7', plot_bgcolor='#f4e8c1',
                  font=dict(family='Georgia, serif'), height=450, barmode='group')
fig.show()

## 6. Distribución Temática por Presidente

Analizamos cómo se distribuyen los temas en cada discurso y cómo varían entre presidentes. Esto revela **prioridades discursivas** de cada gobierno.

In [ ]:
doc_topic_list = []
for i, row in df.iterrows():
    dominant = int(doc_topics[i].argmax())
    doc_topic_list.append({
        'year': int(row['year']),
        'speaker': row['speaker'],
        'topic_dominante': dominant,
        'peso': round(float(doc_topics[i][dominant]), 3),
    })
    for j in range(N_TOPICS):
        doc_topic_list[-1][f'tema_{j}'] = round(float(doc_topics[i][j]), 3)

doc_topics_df = pd.DataFrame(doc_topic_list)
doc_topics_df.head()

In [ ]:
pivot = doc_topics_df.pivot_table(index='speaker', columns='topic_dominante',
                                  values='peso', aggfunc='mean', fill_value=0)

fig = px.imshow(pivot,
                title='Mapa de Calor: Temas Dominantes por Presidente',
                labels=dict(x='Tema', y='Presidente', color='Peso Promedio'),
                aspect='auto',
                color_continuous_scale=['#faf0d7', '#c5a55a', '#6b1d1d'])
fig.update_layout(paper_bgcolor='#faf0d7', plot_bgcolor='#f4e8c1',
                  font=dict(family='Georgia, serif'), height=450)
fig.show()

In [ ]:
all_topics_cols = [f'tema_{j}' for j in range(N_TOPICS)]
topic_stacked = doc_topics_df.groupby('speaker')[all_topics_cols].mean()

topic_stacked.columns = [f'Tema {j}' for j in range(N_TOPICS)]

fig = go.Figure()
colors = ['#c5a55a', '#6b1d1d', '#2d5016', '#1a2744', '#8a2e2e']

for i, col in enumerate(topic_stacked.columns):
    fig.add_trace(go.Bar(
        y=topic_stacked.index, x=topic_stacked[col],
        name=col, orientation='h',
        marker_color=colors[i]
    ))

fig.update_layout(barmode='stack',
                  title='Composición Temática por Presidente',
                  xaxis_title='Proporción del Tema',
                  paper_bgcolor='#faf0d7', plot_bgcolor='#f4e8c1',
                  font=dict(family='Georgia, serif'),
                  legend=dict(traceorder='normal', orientation='h', y=-0.15),
                  height=500)
fig.show()

In [ ]:
fig = px.line(doc_topics_df, x='year', y=[f'tema_{j}' for j in range(N_TOPICS)],
              title='Evolución Temporal de Temas en Discursos Presidenciales',
              labels={'value': 'Peso del Tema', 'variable': 'Tema', 'year': 'Año'},
              markers=True)
fig.update_layout(paper_bgcolor='#faf0d7', plot_bgcolor='#f4e8c1',
                  font=dict(family='Georgia, serif'), height=450,
                  legend=dict(traceorder='normal'))
fig.show()

## 7. Hallazgos Principales

### Temas Dominantes en el Discurso Presidencial Chileno

El análisis revela patrones recurrentes en las Cuentas Públicas presidenciales:

| Período | Tendencias Discursivas |
|---------|----------------------|
| **Siglo XIX** | Consolidación nacional, territorio, soberanía, guerra |
| **Primera mitad del siglo XX** | Reconstrucción, modernización, industria |
| **1960-1990** | Desarrollo social, crisis, derechos humanos, transición democrática |
| **1990-2000** | Integración económica, reformas, reconstrucción post-terremoto |

### Observaciones Clave

1. **Continuidad temática**: Ciertas palabras como "país", "gobierno", "Estado" y "ley" aparecen transversalmente
2. **Evolución semántica**: El contexto geográfico (norte, sur, Santiago) se mantiene como eje articulador
3. **Impacto de eventos**: Terremotos y crisis generan picos temáticos específicos
4. **Transiciones**: Los bigramas revelan cambios de enfoque entre períodos

## 8. Vista Previa del Dashboard

El análisis presentado aquí se complementa con un **dashboard interactivo** de Dash con estética de manuscrito iluminado medieval.

### Características del Dashboard

- **Línea de tiempo interactiva**: Navegación por discursos por año y presidente
- **Análisis NER**: Entidades nombradas extraídas con spaCy
- **Sentimiento**: Análisis de polaridad por discurso
- **Análisis temático**: LDA, bigramas y treemaps

### Ejecución

```bash
# Instalar dependencias
pip install -r requirements.txt

# Ejecutar análisis
python src/analyze_all.py

# Iniciar dashboard
python dashboard.py
# → http://localhost:8051
```

El dashboard utiliza colores inspirados en manuscritos medievales: **parchment** (#f4e8c1), **burgundy** (#6b1d1d) y **gold** (#c5a55a).

## 9. Conclusiones

### Contribuciones

Este análisis demuestra cómo las técnicas de NLP pueden **iluminar patrones ocultos** en el discurso político histórico:

- **LDA** identifica temas latentes sin supervisión humana
- **Bigramas** capturan expresiones discursivas específicas
- **Visualizaciones** hacen accesibles los resultados a públicos no técnicos

### Limitaciones

1. **Corpus reducido**: Solo 14 discursos limitan la generalización estadística
2. **Stopwords manuales**: La lista de stopwords puede omitir o incluir palabras relevantes al contexto chileno
3. **Temporalidad**: Los discursos están espaciados irregularmente en el tiempo
4. **Sesgo de transcripción**: Los PDFs originales pueden tener errores de OCR

### Trabajo Futuro

- **Expandir el corpus**: Incluir todas las Cuentas Públicas disponibles (1832-2026)
- **Modelos transformer**: Utilizar BERT en español para embeddings contextuales
- **Análisis de sentimiento**: Comparar con TextBlob o VADER adaptado al español
- **NER con spaCy**: Extraer entidades geopolíticas chilenas
- **Redes neuronales**: Clasificación automática de discursos por período histórico

---

*Proyecto: Geopolítica Textual NLP — Análisis de Discurso Presidencial Chileno*